# Phase 1 — Karan: Metadata Normalization + Subject x Entity-Class Matrix

**RBI-ObliBench / Agentic RAG compliance system**

Discovers the entity-class and subject-family vocabularies from Akash's committed corpus, normalizes `entity_class`/`subject_family` onto every `DocumentRecord`/`ParagraphRecord`, and builds the full Subject x Entity-Class Matrix. Thin wrapper only — every cell calls into `src/` or `scripts/run_matrix.py`; no pipeline logic lives here.

**Add Input** the `rbi-corpus-v1` Kaggle Dataset (from P1-001) before running — this notebook reads Akash's already-harvested corpus, it does not scrape anything itself.

## 1. Get the code

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/karanLokhande29/Capstone_project.git"
BRANCH = "main"

WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
REPO_DIR = os.path.join(WORKING, "Capstone_project")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # /kaggle/working commonly survives across "Run All" within the same
    # Kaggle session — always sync to the latest commit rather than
    # trusting whatever was checked out last time (see P1-001's history:
    # this exact staleness cost three debugging rounds there).
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True,
).stdout.strip()
print("repository:", REPO_DIR)
print("commit:     ", commit, "-- check this matches the latest commit on GitHub before trusting a run")

## 2. Attach Akash's corpus

This notebook reads `data/metadata/document_manifest.jsonl` and `data/processed/*.jsonl` — either the copies committed/synced into the repo, or (on Kaggle) an attached `rbi-corpus-v1` Dataset mounted under `/kaggle/input/`. If a Dataset is attached, copy its `data/` contents into the writable working root so `scripts/run_matrix.py` can update them in place (never write into the read-only `/kaggle/input` mount).

In [ ]:
import shutil
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DATA = Path(REPO_DIR) / "data"

if KAGGLE_INPUT.is_dir():
    candidates = [d for d in KAGGLE_INPUT.iterdir() if (d / "data").is_dir()]
    if candidates:
        source = candidates[0] / "data"
        print(f"copying corpus data from {source} into the writable working root")
        for sub in ("metadata", "processed", "extracted"):
            src, dst = source / sub, WORKING_DATA / sub
            if src.is_dir():
                dst.mkdir(parents=True, exist_ok=True)
                for f in src.iterdir():
                    if f.is_file():
                        shutil.copy2(f, dst / f.name)
    else:
        print("no attached Dataset with a data/ directory found under /kaggle/input")
else:
    print("not on Kaggle — using the repo's own committed/synced data/ as-is")

manifest = WORKING_DATA / "metadata" / "document_manifest.jsonl"
print(f"document_manifest.jsonl present: {manifest.exists()}", 
      f"({sum(1 for _ in open(manifest)) if manifest.exists() else 0} rows)")

## 3. Unit + integration tests

Unit tests run against fixtures; the integration test runs against the repo's real committed manifest slice — no network calls in any of these.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_metadata_vocabulary.py", "tests/test_matrix_builder.py",
     "tests/test_metadata_matrix_integration.py", "-v"],
    capture_output=True, text=True,
)
print(result.stdout[-6000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
print("TESTS:", "PASS" if result.returncode == 0 else "FAIL")

## 4. Discover vocabularies, normalize, build the matrix

Discovers `entity_class`/`subject_family` vocabularies from the attached corpus, writes normalized values onto `DocumentRecord`/`ParagraphRecord` (never touching `*_raw`), and builds the full Subject x Entity-Class Matrix — one cell per discovered `(entity_class, subject_family)` pair, including unpopulated ones.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/run_matrix.py", "all", "--json"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-4000:])

## 5. Result

Copy into the weekly logbook. Read back through `PathResolver` rather than a hardcoded relative path — see P1-001's notebook for why that distinction matters on Kaggle (working root and the cloned repo directory are not the same path).

In [ ]:
from src.common.config import load_config
from src.common.paths import PathResolver

cfg = load_config()
resolver = PathResolver.from_config(cfg)
report_path = resolver.read_path("reports", "phase1_karan_matrix.md")
print(f"reading: {report_path}\n")
print(report_path.read_text())

---

### Saving the result as a Kaggle Dataset

1. Confirm `data/metadata/` (manifest + vocabularies + unresolved list) and `data/matrix/` are populated under `/kaggle/working/`.
2. **New Dataset** (or **New Version** of `rbi-corpus-v1`) including those, named e.g. `rbi-matrix-v1`.
3. Add the dataset slug to `environment.kaggle.input_datasets` in `config/config.yaml` and commit it, so Meer's notebook (P1-003) can attach it via **Add Input**.
4. Download a representative sample back locally before starting P1-003.